# LeetCode 2923 — Find Champion I bằng Python

Link đề: https://leetcode.com/problems/find-champion-i/ · Độ khó: **Easy** · Chủ đề: Matrix, Tournament, Greedy

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findChampion(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một giải đấu có `n` đội đánh số `0..n-1`; ma trận `grid[i][j] = 1` nghĩa là **đội `i` mạnh hơn đội `j`**. **Nhà vô địch** là đội **không đội nào mạnh hơn nó**.
2. Biết đọc **ma trận kề** (adjacency matrix) theo **hàng** (đội `i` thắng ai) và theo **cột** (ai thắng đội `j`).
3. Đi từ cách **kiểm tra cột của từng đội có toàn số 0 không** → **đếm số trận thắng (tổng hàng) phải bằng `n - 1`** → **loại dần ứng viên bằng một lượt duyệt `O(n)`** → **loại các đội bị đánh bại** → **`max` theo tổng hàng**.
4. Hiểu **vì sao cần điều kiện bắc cầu (transitive)** của đề để cách "ứng viên" `O(n)` đúng.
5. Tránh các lỗi kinh điển: nhầm hàng với cột, trả về **số trận thắng** thay vì **chỉ số đội**, kiểm tra cả đường chéo `grid[i][i]`, và đảo chiều so sánh khi loại ứng viên.

## 2. Đề bài & ví dụ

Có `n` đội đánh số từ `0` đến `n - 1`. Cho ma trận nhị phân `grid` kích thước `n x n`:

- `grid[i][j] == 1` nghĩa là **đội `i` mạnh hơn đội `j`**;
- `grid[i][j] == 0` nghĩa là đội `i` **không** mạnh hơn đội `j`.

Một đội `a` là **nhà vô địch** nếu **không có đội `b` nào mạnh hơn `a`**. Hãy trả về **chỉ số** của nhà vô địch.

Đề đảm bảo: với mọi `i`: `grid[i][i] == 0`; với mọi `i != j`: **đúng một** trong `grid[i][j]`, `grid[j][i]` bằng `1` (hai đội luôn phân định mạnh yếu); và có **tính bắc cầu**: nếu đội `a` mạnh hơn `b` và `b` mạnh hơn `c` thì `a` mạnh hơn `c`. Do đó **luôn có đúng một nhà vô địch**.

**Ví dụ 1**

- Input: `grid = [[0, 1], [0, 0]]`
- Output: `0`
- Giải thích: có hai đội; đội `0` mạnh hơn đội `1` (`grid[0][1] = 1`). Không đội nào mạnh hơn đội `0`, nên đội `0` là nhà vô địch.

**Ví dụ 2**

- Input: `grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]`
- Output: `1`
- Giải thích: đội `1` mạnh hơn đội `0` và đội `2` (`grid[1][0] = grid[1][2] = 1`); đội `0` mạnh hơn đội `2`. Không đội nào mạnh hơn đội `1`, nên đáp án là `1`.

**Ràng buộc:** `n == grid.length == grid[i].length`; `2 <= n <= 100`; `grid[i][j]` là `0` hoặc `1`; `grid[i][i] == 0`; `grid[i][j] != grid[j][i]` khi `i != j`; quan hệ bắc cầu; luôn tồn tại nhà vô địch.

In [1]:
from typing import List
import random

examples = [
    ([[0, 1], [0, 0]], 0),
    ([[0, 0, 1], [1, 0, 1], [0, 0, 0]], 1),
]
for grid, expected in examples:
    print("grid =", grid, "-> kỳ vọng", expected)

grid = [[0, 1], [0, 0]] -> kỳ vọng 0
grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Ma trận hai chiều `grid[i][j]`

`grid` là **danh sách các danh sách**: `grid[i]` là **hàng `i`** (một danh sách), `grid[i][j]` là phần tử ở **hàng `i`, cột `j`**. Ở bài này: **hàng `i`** cho biết đội `i` **thắng** những đội nào (`grid[i][j] == 1`); **cột `j`** cho biết những đội nào **thắng đội `j`** (`grid[i][j] == 1` với các `i`).

In [2]:
grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]
print("hàng 1 :", grid[1])                       # [1, 0, 1]: đội 1 thắng đội 0 và đội 2
print("grid[1][2] =", grid[1][2])                # 1
col2 = [row[2] for row in grid]
print("cột 2  :", col2)                          # [1, 1, 0]: đội 0 và đội 1 thắng đội 2

hàng 1 : [1, 0, 1]
grid[1][2] = 1
cột 2  : [1, 1, 0]


### 3.2. Lấy một cột và duyệt ma trận

Python không có cú pháp lấy cột trực tiếp. Ta dùng **list comprehension** `[row[j] for row in grid]`, hoặc `zip(*grid)` để **chuyển vị** (mỗi phần tử của `zip(*grid)` là một cột). `len(grid)` là số hàng (cũng là số đội `n`).

In [3]:
print([row[0] for row in grid])                  # cột 0
for j, col in enumerate(zip(*grid)):              # từng cột một
    print(f"cột {j}:", list(col))

[0, 1, 0]
cột 0: [0, 1, 0]
cột 1: [0, 0, 0]
cột 2: [1, 1, 0]


### 3.3. Đội vô địch trong ngôn ngữ ma trận

"Không đội nào mạnh hơn đội `a`" nghĩa là **không có `i`** nào với `grid[i][a] == 1`, tức **cột `a` toàn số 0**. Vì mỗi cặp đội phân định mạnh yếu, một đội vô địch cũng **mạnh hơn mọi đội khác** (`grid[a][j] == 1` với mọi `j != a`), tức **hàng `a` có đúng `n - 1` số 1**.

In [4]:
n = len(grid)
for a in range(n):
    column_all_zero = all(grid[i][a] == 0 for i in range(n))
    wins = sum(grid[a])
    print(f"đội {a}: cột toàn 0? {column_all_zero} | số trận thắng = {wins} (n - 1 = {n - 1})")

đội 0: cột toàn 0? False | số trận thắng = 1 (n - 1 = 2)
đội 1: cột toàn 0? True | số trận thắng = 2 (n - 1 = 2)
đội 2: cột toàn 0? False | số trận thắng = 0 (n - 1 = 2)


### 3.4. `sum`, `all`, `any` trên danh sách

- `sum(row)`: cộng các phần tử (mỗi `1` là một trận thắng);
- `all(cond for ...)`: `True` nếu **mọi** phần tử thỏa;
- `any(cond for ...)`: `True` nếu **có ít nhất một** phần tử thỏa.

Ví dụ "không ai thắng đội `a`" ⇔ `not any(grid[i][a] for i in range(n))` ⇔ `all(grid[i][a] == 0 ...)`.

In [5]:
row = [1, 0, 1]
print(sum(row), all(v == 1 for v in row), any(v == 1 for v in row))   # 2 False True
print(all([]), any([]))                                                # True False (dãy rỗng)

2 False True
True False


### 3.5. Tính bắc cầu (transitive) và vì sao hữu ích

Bắc cầu: nếu `a` thắng `b` và `b` thắng `c` thì `a` thắng `c`. Hệ quả: các đội có thể **xếp thứ hạng tuyến tính** từ mạnh nhất đến yếu nhất (không có vòng "kéo búa bao": `a` thắng `b`, `b` thắng `c`, `c` thắng `a` bị loại trừ). Nhờ đó đội mạnh nhất tồn tại và **duy nhất**, và có thể tìm nhanh bằng cách **loại dần**.

In [6]:
# Dựng một giải đấu bắc cầu từ thứ hạng: rank nhỏ = mạnh hơn
rank = [2, 0, 1]                                  # đội 1 mạnh nhất, đội 2 nhì, đội 0 yếu nhất
n = len(rank)
grid_from_rank = [[1 if rank[i] < rank[j] else 0 for j in range(n)] for i in range(n)]
print(grid_from_rank)                             # [[0, 0, 0], [1, 0, 1], [1, 0, 0]]
print("đội mạnh nhất:", rank.index(0))

[[0, 0, 0], [1, 0, 1], [1, 0, 0]]
đội mạnh nhất: 1


### 3.6. Ý tưởng "ứng viên" (candidate) — giữ một đội còn hy vọng

Ý tưởng cổ điển: giữ một **ứng viên** `c`. Xét lần lượt đội `i`:

- nếu **`i` mạnh hơn `c`** (`grid[i][c] == 1`) thì `c` **chắc chắn không phải nhà vô địch** (có người hơn). Đổi ứng viên thành `i`;
- ngược lại, vì mọi cặp đều phân định mạnh yếu, **`c` mạnh hơn `i`**, nên `i` **chắc chắn không phải nhà vô địch**: giữ nguyên `c`.

Mỗi lần xét một đội ta **loại đúng một đội** khỏi danh sách hy vọng (`c` cũ hoặc `i`). Sau `n - 1` lần chỉ còn một đội.

In [7]:
grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]
c = 0
for i in range(1, len(grid)):
    if grid[i][c] == 1:
        print(f"đội {i} thắng đội {c} -> loại {c}, ứng viên mới = {i}")
        c = i
    else:
        print(f"đội {c} thắng đội {i} -> loại {i}, ứng viên vẫn = {c}")
print("ứng viên cuối:", c)

đội 1 thắng đội 0 -> loại 0, ứng viên mới = 1
đội 1 thắng đội 2 -> loại 2, ứng viên vẫn = 1
ứng viên cuối: 1


### 3.7. `max` với `key` để chọn theo một tiêu chí

`max(iterable, key=hàm)` trả về phần tử có **`key` lớn nhất**. Với `max(range(n), key=lambda i: sum(grid[i]))` ta lấy **đội có nhiều trận thắng nhất**, và trả về **chỉ số đội** (không phải số trận thắng).

In [8]:
wins = [sum(row) for row in grid]
print("số trận thắng:", wins)                                    # [1, 2, 0]
print("đội thắng nhiều nhất:", max(range(len(grid)), key=lambda i: sum(grid[i])))   # 1
print("(đây là giá trị thắng):", max(wins), "-> không phải đáp án")

số trận thắng: [1, 2, 0]
đội thắng nhiều nhất: 1
(đây là giá trị thắng): 2 -> không phải đáp án


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nhìn bảng và hỏi cho mỗi đội: **có ai thắng nó không?** Với `grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]`:

| đội `a` | cột `a` (ai thắng `a`?) | có ai thắng `a`? | hàng `a` (`a` thắng ai?) | số trận thắng |
|---|---|---|---|---|
| 0 | `[0, 1, 0]` | **có** (đội 1) | `[0, 0, 1]` | 1 |
| 1 | `[0, 0, 0]` | **không** | `[1, 0, 1]` | **2 = n - 1** |
| 2 | `[1, 1, 0]` | **có** (đội 0, 1) | `[0, 0, 0]` | 0 |

Chỉ đội `1` có cột toàn `0` và cũng thắng đủ `n - 1 = 2` đội. Đáp án là `1`. Thuật toán: **tìm đội mà cột của nó không có số 1 nào** (hoặc đội thắng đủ `n - 1` đội).

## 5. Cách 1 — Brute Force / baseline: kiểm tra cột của từng đội

**Ý tưởng.** Với mỗi đội `a`, duyệt mọi đội `i` và kiểm tra `grid[i][a]`: nếu có `1` thì `a` bị đánh bại; nếu không có thì `a` là nhà vô địch. Ghi đúng điều kiện của đề.

In [9]:
def champion_column_scan(grid: List[List[int]]) -> int:
    n = len(grid)
    for a in range(n):
        beaten = False
        for i in range(n):
            if grid[i][a] == 1:               # đội i mạnh hơn đội a
                beaten = True
                break
        if not beaten:                         # không ai mạnh hơn a
            return a
    return -1                                  # lý thuyết không xảy ra (đề đảm bảo có đáp án)


for grid, expected in examples:
    print(grid, "->", champion_column_scan(grid), "(kỳ vọng", expected, ")")

[[0, 1], [0, 0]] -> 0 (kỳ vọng 0 )
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1 (kỳ vọng 1 )


In [10]:
# Theo dõi từng đội của ví dụ 2
grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]
n = len(grid)
for a in range(n):
    beaters = [i for i in range(n) if grid[i][a] == 1]
    print(f"đội {a}: bị các đội {beaters} đánh bại ->", "KHÔNG phải vô địch" if beaters else "VÔ ĐỊCH")

đội 0: bị các đội [1] đánh bại -> KHÔNG phải vô địch
đội 1: bị các đội [] đánh bại -> VÔ ĐỊCH
đội 2: bị các đội [0, 1] đánh bại -> KHÔNG phải vô địch


**Độ phức tạp (Cách 1).** Gọi `n` là số đội. Mỗi đội quét cột `O(n)`: **Time `O(n²)`**; chỉ vài biến: **Space `O(1)`**.

**Nút thắt.** Phải duyệt gần như cả ma trận (`n²` ô). Nhưng đề cho **nhiều thông tin** (mọi cặp đều phân định, bắc cầu); ta có thể tìm nhà vô địch chỉ bằng `O(n)` lần tra ô.

## 6. Ý tưởng thuật toán / bước đột phá

Dùng **tính phân định + bắc cầu** để **loại ứng viên**: giữ một ứng viên `c`, xét `i = 1..n-1`; nếu `grid[i][c] == 1` thì chuyển ứng viên sang `i`. Cuối cùng `c` là nhà vô địch.

**Vì sao đúng (phác họa).** Mỗi bước loại được một đội **chắc chắn không phải** nhà vô địch (đội thua trong cặp `(c, i)`). Sau `n - 1` bước, `n - 1` đội bị loại và còn **một** đội; vì đề đảm bảo có nhà vô địch (mà nó không bao giờ bị loại, vì nó không thua trận nào), đội còn lại chính là nó.

Bài này chỉ cần `n - 1` lần tra ô: **`O(n)`**.

## 7. Các cách tiếp theo

### Cách 2 — Đếm số trận thắng (tổng hàng) phải bằng `n - 1`

**Giải thích.** Nhà vô địch thắng **mọi** đội khác nên `sum(grid[a]) == n - 1`. (Đường chéo `grid[a][a] = 0`.) Duyệt các đội, trả đội có tổng hàng bằng `n - 1`. Cách này **dùng tính phân định** của đề (mỗi cặp có người thắng).

In [11]:
def champion_row_sum(grid: List[List[int]]) -> int:
    n = len(grid)
    for a in range(n):
        if sum(grid[a]) == n - 1:              # thắng đủ n - 1 đội khác
            return a
    return -1


for grid, expected in examples:
    print(grid, "->", champion_row_sum(grid), "(kỳ vọng", expected, ")")

[[0, 1], [0, 0]] -> 0 (kỳ vọng 0 )
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1 (kỳ vọng 1 )


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def findChampion(self, grid: List[List[int]]) -> int:
        n = len(grid)
        for a in range(n):
            if sum(grid[a]) == n - 1:
                return a
        return -1

**Độ phức tạp Cách 2:** **Time `O(n²)`** (cộng mọi hàng), **Space `O(1)`**.

### Cách 3 — Ứng viên: loại dần trong một lượt `O(n)`

**Giải thích.** Giữ `champ`; với mỗi `i`, nếu `grid[i][champ] == 1` (đội `i` mạnh hơn ứng viên hiện tại) thì `champ = i`. Chỉ `n - 1` lần tra ô.

In [13]:
def champion_candidate(grid: List[List[int]]) -> int:
    champ = 0                                  # ứng viên hiện tại
    for i in range(1, len(grid)):
        if grid[i][champ] == 1:                # i mạnh hơn ứng viên -> ứng viên cũ bị loại
            champ = i
    return champ


for grid, expected in examples:
    print(grid, "->", champion_candidate(grid), "(kỳ vọng", expected, ")")

[[0, 1], [0, 0]] -> 0 (kỳ vọng 0 )
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1 (kỳ vọng 1 )


In [14]:
# Code nộp LeetCode cho Cách 3
from typing import List

class Solution:
    def findChampion(self, grid: List[List[int]]) -> int:
        champ = 0
        for i in range(1, len(grid)):
            if grid[i][champ] == 1:
                champ = i
        return champ

**Độ phức tạp Cách 3:** **Time `O(n)`** (đúng `n - 1` lần tra ô), **Space `O(1)`**.

### Cách 4 — Tập các đội "chưa bị đánh bại"

**Giải thích.** Bắt đầu với tập mọi đội. Duyệt mọi cặp `(i, j)`; nếu `grid[i][j] == 1` thì **đội `j` bị đánh bại**, loại `j` khỏi tập. Cuối cùng tập còn đúng một đội. Cách này **không cần bắc cầu hay phân định**: chỉ dựa vào định nghĩa (không ai mạnh hơn).

In [15]:
def champion_unbeaten_set(grid: List[List[int]]) -> int:
    n = len(grid)
    unbeaten = set(range(n))                   # mọi đội ban đầu còn "chưa bị đánh bại"
    for i in range(n):
        for j in range(n):
            if grid[i][j] == 1:
                unbeaten.discard(j)            # j có người thắng -> loại khỏi tập
    assert len(unbeaten) == 1                  # đề đảm bảo đúng một nhà vô địch
    return next(iter(unbeaten))


for grid, expected in examples:
    print(grid, "->", champion_unbeaten_set(grid), "(kỳ vọng", expected, ")")

[[0, 1], [0, 0]] -> 0 (kỳ vọng 0 )
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n²)`**, **Space `O(n)`** (tập các đội).

### Cách 5 — `max` theo số trận thắng

**Giải thích.** Nhà vô địch có số trận thắng nhiều nhất (`n - 1`), các đội khác ít hơn (vì thứ hạng tuyến tính: đội hạng `r` thắng đúng `n - 1 - r` đội). Dùng `max(range(n), key=lambda i: sum(grid[i]))`, nhớ trả **chỉ số đội**.

In [16]:
def champion_max_wins(grid: List[List[int]]) -> int:
    return max(range(len(grid)), key=lambda i: sum(grid[i]))


for grid, expected in examples:
    print(grid, "->", champion_max_wins(grid), "(kỳ vọng", expected, ")")

[[0, 1], [0, 0]] -> 0 (kỳ vọng 0 )
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n²)`**, **Space `O(1)`**.

> Cách 3 (`O(n)`) là cách nên nộp nếu bạn nắm chắc lý do; Cách 1 và 2 dễ chứng minh. Mọi cách đều nhanh với `n <= 100`.

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 (ứng viên) trên giải đấu 5 đội; (b) Cách 4 (loại các đội bị đánh bại) trên ví dụ 2.

In [17]:
def make_grid_from_rank(rank):
    """rank[i] nhỏ hơn = đội i mạnh hơn. Trả về ma trận bắc cầu."""
    n = len(rank)
    return [[1 if rank[i] < rank[j] else 0 for j in range(n)] for i in range(n)]


def trace_candidate(grid):
    champ = 0
    print("n =", len(grid), "| bắt đầu ứng viên =", champ)
    for i in range(1, len(grid)):
        if grid[i][champ] == 1:
            print(f"i={i}: grid[{i}][{champ}]=1 -> đội {i} mạnh hơn đội {champ}; ứng viên đổi {champ} -> {i}")
            champ = i
        else:
            print(f"i={i}: grid[{i}][{champ}]=0 -> đội {champ} mạnh hơn đội {i}; giữ ứng viên {champ}")
    print("=> nhà vô địch =", champ)
    return champ


rank5 = [3, 0, 4, 1, 2]                      # đội 1 mạnh nhất, rồi đội 3, đội 4, đội 0, đội 2
grid5 = make_grid_from_rank(rank5)
for row in grid5:
    print(row)
assert trace_candidate(grid5) == 1

[0, 0, 1, 0, 0]
[1, 0, 1, 1, 1]
[0, 0, 0, 0, 0]
[1, 0, 1, 0, 1]
[1, 0, 1, 0, 0]
n = 5 | bắt đầu ứng viên = 0
i=1: grid[1][0]=1 -> đội 1 mạnh hơn đội 0; ứng viên đổi 0 -> 1
i=2: grid[2][1]=0 -> đội 1 mạnh hơn đội 2; giữ ứng viên 1
i=3: grid[3][1]=0 -> đội 1 mạnh hơn đội 3; giữ ứng viên 1
i=4: grid[4][1]=0 -> đội 1 mạnh hơn đội 4; giữ ứng viên 1
=> nhà vô địch = 1


In [18]:
def trace_unbeaten(grid):
    n = len(grid)
    unbeaten = set(range(n))
    print("ban đầu:", sorted(unbeaten))
    for i in range(n):
        for j in range(n):
            if grid[i][j] == 1:
                if j in unbeaten:
                    unbeaten.discard(j)
                    print(f"grid[{i}][{j}]=1 -> đội {i} thắng đội {j}: loại {j}; còn {sorted(unbeaten)}")
    print("=> nhà vô địch =", next(iter(unbeaten)))
    return next(iter(unbeaten))


assert trace_unbeaten([[0, 0, 1], [1, 0, 1], [0, 0, 0]]) == 1

ban đầu: [0, 1, 2]
grid[0][2]=1 -> đội 0 thắng đội 2: loại 2; còn [0, 1]
grid[1][0]=1 -> đội 1 thắng đội 0: loại 0; còn [1]
=> nhà vô địch = 1


Bảng tóm tắt (Cách 3, `rank = [3, 0, 4, 1, 2]`):

| `i` | `champ` trước | `grid[i][champ]` | hành động | `champ` sau |
|---|---|---|---|---|
| 1 | 0 | 1 (đội 1 mạnh hơn đội 0) | đổi | 1 |
| 2 | 1 | 0 | giữ | 1 |
| 3 | 1 | 0 | giữ | 1 |
| 4 | 1 | 0 | giữ | 1 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (ứng viên `O(n)`). Đúng class `Solution`, đúng method `findChampion(self, grid: List[List[int]]) -> int`.

In [19]:
from typing import List

class Solution:
    def findChampion(self, grid: List[List[int]]) -> int:
        champ = 0                                # ứng viên hiện tại
        for i in range(1, len(grid)):
            if grid[i][champ] == 1:              # đội i mạnh hơn ứng viên -> thay ứng viên
                champ = i
        return champ


sol = Solution()
for grid, expected in examples:
    print(grid, "->", sol.findChampion(grid))

[[0, 1], [0, 0]] -> 0
[[0, 0, 1], [1, 0, 1], [0, 0, 0]] -> 1


## 10. Vì sao thuật toán đúng

- **Mỗi bước loại một đội "chắc chắn thua":** xét cặp `(champ, i)`. Đề đảm bảo đúng một trong `grid[i][champ]` và `grid[champ][i]` bằng `1`. Nếu `grid[i][champ] == 1` thì `i` mạnh hơn `champ`, nên `champ` có người hơn, **không phải** nhà vô địch. Ngược lại `champ` mạnh hơn `i`, nên `i` **không phải** nhà vô địch.
- **Nhà vô địch không bao giờ bị loại:** gọi `w` là nhà vô địch (không ai mạnh hơn). Khi `w` đang là ứng viên, không có `i` nào làm `grid[i][w] == 1`, nên `w` **giữ nguyên**. Khi `w` được xét như `i` (với ứng viên `champ != w`): `grid[w][champ] == 1` vì `w` mạnh hơn mọi đội khác (không ai hơn `w`, và mỗi cặp phân định), nên `champ` được thay bằng `w`. Từ lúc đó `w` luôn là ứng viên.
- **Kết quả:** sau lượt duyệt `w` là ứng viên cuối cùng. (Nếu `w` đúng là đội `0`, nó là ứng viên ngay từ đầu và không bị đổi.)
- **Vai trò của giả thiết:** "mỗi cặp phân định" bảo đảm `w` mạnh hơn mọi đội khác; "đề đảm bảo có nhà vô địch" bảo đảm `w` tồn tại. (Bắc cầu bảo đảm tính duy nhất và tính hợp lý của dữ liệu.)
- **Tính chất kiểm được:** kết quả có cột toàn `0` và hàng có đúng `n - 1` số `1`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số đội (số hàng, số cột).

- **Cách 1, 2, 4, 5:** Time `O(n²)` (đọc gần hết ma trận), Space `O(1)` (Cách 4: `O(n)`).
- **Cách 3 (ứng viên):** Time **`O(n)`** (đúng `n - 1` ô được đọc), Space **`O(1)`**.
- Ma trận có `n²` ô nhưng ta chỉ cần `O(n)` ô: nhờ ràng buộc "mọi cặp phân định" mà một ô `grid[i][champ]` đủ cho cả hai chiều của cặp `(champ, i)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`, kiểm tra ví dụ, biên, rồi sinh giải đấu bắc cầu ngẫu nhiên từ một thứ hạng.

In [20]:
sol = Solution()
impls = {
    "column_scan": champion_column_scan,
    "row_sum": champion_row_sum,
    "candidate": champion_candidate,
    "unbeaten_set": champion_unbeaten_set,
    "max_wins": champion_max_wins,
    "submit": sol.findChampion,
}

for grid, expected in examples:
    for key, fn in impls.items():
        assert fn([row[:] for row in grid]) == expected, (key, grid, fn(grid), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def check_all(grid, expected, label):
    for key, fn in impls.items():
        got = fn([row[:] for row in grid])
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([[0, 1], [0, 0]], 0, "n = 2, đội 0 mạnh hơn")
check_all([[0, 0], [1, 0]], 1, "n = 2, đội 1 mạnh hơn")
check_all([[0, 1, 1], [0, 0, 1], [0, 0, 0]], 0, "đội 0 mạnh nhất, xếp tăng dần")
check_all([[0, 0, 0], [1, 0, 0], [1, 1, 0]], 2, "đội cuối mạnh nhất")
check_all(make_grid_from_rank([1, 2, 0]), 2, "thứ hạng [1, 2, 0]: đội 2 mạnh nhất")
check_all(make_grid_from_rank([4, 3, 2, 1, 0]), 4, "đội cuối mạnh nhất (n = 5)")
check_all(make_grid_from_rank([0, 1, 2, 3, 4]), 0, "đội đầu mạnh nhất (n = 5)")

PASS n = 2, đội 0 mạnh hơn -> 0
PASS n = 2, đội 1 mạnh hơn -> 1
PASS đội 0 mạnh nhất, xếp tăng dần -> 0
PASS đội cuối mạnh nhất -> 2
PASS thứ hạng [1, 2, 0]: đội 2 mạnh nhất -> 2
PASS đội cuối mạnh nhất (n = 5) -> 4
PASS đội đầu mạnh nhất (n = 5) -> 0


### 12.2. Tính chất và phản ví dụ

In [22]:
# Tính chất: nhà vô địch có cột toàn 0 và hàng có n - 1 số 1; duy nhất
random.seed(23)
for _ in range(200):
    n = random.randint(2, 30)
    rank = list(range(n))
    random.shuffle(rank)
    grid = make_grid_from_rank(rank)
    champ = sol.findChampion(grid)
    assert all(grid[i][champ] == 0 for i in range(n))      # không ai mạnh hơn
    assert sum(grid[champ]) == n - 1                       # mạnh hơn mọi đội khác
    assert rank[champ] == 0
    others = [a for a in range(n) if a != champ]
    assert all(any(grid[i][a] == 1 for i in range(n)) for a in others)    # mọi đội khác đều bị ai đó thắng
print("PASS nhà vô địch: cột toàn 0, hàng n - 1 số 1, duy nhất")

# Điều kiện của đề trên ma trận sinh ra (kiểm tra dữ liệu sinh đúng)
def valid_tournament(grid):
    n = len(grid)
    if any(grid[i][i] != 0 for i in range(n)):
        return False
    if any(grid[i][j] == grid[j][i] for i in range(n) for j in range(n) if i != j):
        return False
    return all(grid[a][c] == 1 for a in range(n) for b in range(n) for c in range(n)
               if grid[a][b] == 1 and grid[b][c] == 1)

assert valid_tournament(make_grid_from_rank([2, 0, 1, 3]))
print("PASS ma trận sinh ra thỏa các điều kiện của đề")

# Đầu vào không bị sửa
grid = make_grid_from_rank([3, 0, 4, 1, 2])
copy = [row[:] for row in grid]
for key, fn in impls.items():
    fn(grid)
assert grid == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: trả về số trận thắng thay vì chỉ số đội
def sai_tra_so_tran(grid):
    return max(sum(row) for row in grid)

assert sai_tra_so_tran([[0, 0, 1], [1, 0, 1], [0, 0, 0]]) == 2 != 1
print("PASS phản ví dụ: trả", sai_tra_so_tran([[0, 0, 1], [1, 0, 1], [0, 0, 0]]), "(số trận thắng) thay vì 1 (chỉ số)")

# Phản ví dụ 2: đảo chiều, tìm hàng toàn 0 (đội yếu nhất)
def sai_hang_toan_0(grid):
    for a in range(len(grid)):
        if sum(grid[a]) == 0:
            return a

assert sai_hang_toan_0([[0, 0, 1], [1, 0, 1], [0, 0, 0]]) == 2 != 1
print("PASS phản ví dụ: hàng toàn 0 là đội yếu nhất (2), không phải nhà vô địch (1)")

PASS nhà vô địch: cột toàn 0, hàng n - 1 số 1, duy nhất
PASS ma trận sinh ra thỏa các điều kiện của đề
PASS đầu vào không bị sửa
PASS phản ví dụ: trả 2 (số trận thắng) thay vì 1 (chỉ số)
PASS phản ví dụ: hàng toàn 0 là đội yếu nhất (2), không phải nhà vô địch (1)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
random.seed(2923)
for trial in range(1000):
    n = random.randint(2, 100)
    rank = list(range(n))
    random.shuffle(rank)
    grid = make_grid_from_rank(rank)
    expected = rank.index(0)
    for key, fn in impls.items():
        assert fn([row[:] for row in grid]) == expected, (key, n)
print("PASS 1000 giải đấu ngẫu nhiên (n tới 100) - mọi cách khớp nhau")

PASS 1000 giải đấu ngẫu nhiên (n tới 100) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm hàng với cột

In [24]:
def sai_hang_cot(grid):
    n = len(grid)
    for a in range(n):
        if all(grid[a][j] == 0 for j in range(n)):      # kiểm tra HÀNG a toàn 0 (đội a không thắng ai)
            return a


grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]
print("sai  :", sai_hang_cot(grid), "(đội thua mọi trận) | đúng:", Solution().findChampion(grid))

sai  : 2 (đội thua mọi trận) | đúng: 1


**Vì sao sai:** `grid[i][j] == 1` nghĩa là **`i` mạnh hơn `j`**. Hàng `a` toàn `0` nghĩa là `a` **không thắng ai** (đội yếu nhất). Nhà vô địch là đội **không ai thắng nó**: **cột** `a` toàn `0` (`grid[i][a] == 0` với mọi `i`).

### Lỗi 2: Trả về số trận thắng thay vì chỉ số đội

In [25]:
def sai_tra_gia_tri(grid):
    return max(sum(row) for row in grid)


grid = [[0, 1], [0, 0]]
print("sai  :", sai_tra_gia_tri(grid), "| đúng:", Solution().findChampion(grid))

sai  : 1 | đúng: 0


**Vì sao sai:** `max(sum(row) for row in grid)` cho **giá trị lớn nhất** (số trận thắng) chứ không phải **chỉ số** của đội đó. Dùng `max(range(n), key=...)` hoặc `argmax` để lấy chỉ số.

### Lỗi 3: Kiểm tra cả ô đường chéo `grid[i][i]`

In [26]:
def sai_dem_n(grid):
    n = len(grid)
    for a in range(n):
        if sum(grid[a]) == n:                           # đòi thắng n đội: kể cả chính mình (không thể)
            return a
    return -1


grid = [[0, 1], [0, 0]]
print("sai  :", sai_dem_n(grid), "| đúng:", Solution().findChampion(grid))

sai  : -1 | đúng: 0


**Vì sao sai:** `grid[i][i] == 0` nên một đội thắng tối đa `n - 1` đội (không thắng chính mình). Điều kiện đúng là `sum(grid[a]) == n - 1`.

### Lỗi 4: Đảo chiều khi loại ứng viên

In [27]:
def sai_dao_chieu(grid):
    champ = 0
    for i in range(1, len(grid)):
        if grid[champ][i] == 1:                         # ngược: đọc hàng thay vì cột
            champ = i
    return champ


grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]
print("sai  :", sai_dao_chieu(grid), "| đúng:", Solution().findChampion(grid))

sai  : 2 | đúng: 1


**Vì sao sai:** ta cần chuyển ứng viên khi **`i` mạnh hơn `champ`**, tức `grid[i][champ] == 1`. Viết `grid[champ][i] == 1` (champ mạnh hơn `i`) làm ngược lại: ứng viên chuyển sang đội **yếu** hơn.

### Lỗi 5: Vòng lặp ứng viên bỏ sót đội cuối

In [28]:
def sai_bo_sot_cuoi(grid):
    champ = 0
    for i in range(1, len(grid) - 1):                   # thiếu đội cuối: range dừng ở n - 2
        if grid[i][champ] == 1:
            champ = i
    return champ


grid = [[0, 0, 0], [1, 0, 0], [1, 1, 0]]               # đội 2 mạnh nhất
print("sai  :", sai_bo_sot_cuoi(grid), "| đúng:", Solution().findChampion(grid))

sai  : 1 | đúng: 2


**Vì sao sai:** đội cuối có chỉ số `n - 1`, nên vòng lặp phải chạy tới `range(1, n)` (gồm `n - 1`). Viết `range(1, n - 1)` bỏ sót đội cuối, đúng lúc nó là nhà vô địch. Lỗi "off-by-one" như vậy thường lộ ra khi thử **đội cuối là nhà vô địch**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "giải đấu, ai mạnh hơn ai, tìm đội vô địch / không bị ai thắng" → **cột toàn 0** (không bị thắng) hoặc **hàng đủ `n - 1`**.
- Với quan hệ **phân định + bắc cầu** → dùng **loại dần ứng viên**, `O(n)`.
- Họ bài "Find the Celebrity" (người nổi tiếng: ai cũng biết, nó không biết ai) dùng cùng kỹ thuật ứng viên.
- Luôn xác định rõ **hàng/cột** nghĩa là gì trước khi viết code: hãy ghi lên giấy "`grid[i][j] = 1` nghĩa là `i` ... `j`".
- Trả về **chỉ số**, không phải giá trị (`max` vs `argmax`).
- Kiểm chứng bằng cách sinh dữ liệu từ một thứ hạng (rank) ngẫu nhiên.

## 15. Biến thể của bài

1. **Find Champion II** (cùng đợt): giải đấu cho dưới dạng danh sách cạnh có hướng, có thể **không có** nhà vô địch hoặc nhiều ứng viên; đáp án là đội có **in-degree 0** nếu duy nhất, ngược lại `-1`.
2. Trả về **thứ hạng đầy đủ** của mọi đội (sắp xếp theo số trận thắng).
3. Tìm đội **yếu nhất** (không thắng ai): hàng toàn `0`.
4. **Find the Celebrity** (ai cũng biết người đó, người đó không biết ai).
5. Khi quan hệ **không bắc cầu** (có vòng): có thể không có nhà vô địch; cần kiểm tra bằng đếm cột.
6. Đếm **số đội không bị ai thắng** (ứng viên vô địch) trong ma trận tùy ý.

Ta thử biến thể 2, 3 và 6:

In [29]:
def full_ranking(grid):
    """Biến thể 2: sắp xếp đội theo số trận thắng giảm dần (đúng khi bắc cầu)."""
    return sorted(range(len(grid)), key=lambda i: -sum(grid[i]))


def weakest(grid):
    """Biến thể 3: đội không thắng ai (hàng toàn 0)."""
    return next(i for i in range(len(grid)) if sum(grid[i]) == 0)


def count_unbeaten(grid):
    """Biến thể 6: số đội mà cột toàn 0 (với ma trận tùy ý)."""
    n = len(grid)
    return sum(1 for a in range(n) if all(grid[i][a] == 0 for i in range(n)))


g = make_grid_from_rank([3, 0, 4, 1, 2])
print("2. xếp hạng :", full_ranking(g))              # [1, 3, 4, 0, 2]
print("3. yếu nhất :", weakest(g))                   # 2
print("6. số đội không bị thắng:", count_unbeaten(g), "| với ma trận không phân định:", count_unbeaten([[0, 0], [0, 0]]))
assert full_ranking(g) == [1, 3, 4, 0, 2] and weakest(g) == 2

2. xếp hạng : [1, 3, 4, 0, 2]
3. yếu nhất : 2
6. số đội không bị thắng: 1 | với ma trận không phân định: 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra cột toàn `0` | không ai thắng đội `a` | `O(n²)` | `O(1)` | đúng theo định nghĩa; không cần giả thiết nào |
| 2. Tổng hàng `== n - 1` | thắng mọi đội khác | `O(n²)` | `O(1)` | dùng tính phân định của đề |
| 3. Ứng viên một lượt | loại dần, mỗi bước 1 ô | `O(n)` | `O(1)` | **cách nộp bài**, tối ưu |
| 4. Tập "chưa bị đánh bại" | loại `j` khi `grid[i][j] == 1` | `O(n²)` | `O(n)` | tổng quát (không cần bắc cầu) |
| 5. `max` theo tổng hàng | đội thắng nhiều nhất | `O(n²)` | `O(1)` | ngắn gọn (nhớ trả chỉ số) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Với `grid = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]`, không chạy code, đoán: `grid[1][2]`, `[row[0] for row in grid]`, `sum(grid[1])`. Đoán thêm: `Solution().findChampion([[0, 0], [1, 0]])` và `Solution().findChampion([[0, 1, 1], [0, 0, 1], [0, 0, 0]])`.

**Bài 2 (tính tay).** Với `grid = [[0, 1, 1, 1], [0, 0, 1, 1], [0, 0, 0, 1], [0, 0, 0, 0]]`: lập bảng cột và tổng hàng của từng đội; tìm nhà vô địch; mô phỏng Cách 3 từng bước.

**Bài 3 (điền TODO).** Hoàn thành `champion_todo(grid)` theo Cách 3.

In [30]:
def champion_todo(grid):
    champ = 0
    # TODO 1: duyệt i từ 1 đến len(grid) - 1
    # TODO 2: nếu grid[i][champ] == 1 thì champ = i
    # TODO 3: return champ
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `champion_column_scan2(grid)` bằng cách kiểm tra cột (Cách 1) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `weakest_team(grid)` trả về đội yếu nhất (hàng toàn `0`) bằng **kỹ thuật ứng viên `O(n)`**. (b) Viết `find_champion_or_minus_one(grid)` cho ma trận **tùy ý** (không đảm bảo giả thiết): trả về nhà vô địch nếu **có đúng một** đội không bị ai thắng, ngược lại `-1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `[0, 1, 0]`, `2`, `1`, `0`. Bài 2: viết từng cột `[grid[i][a] for i in range(4)]`.
- **Mức 2 (vừa):** Bài 2: cột 0 `[0,0,0,0]` (toàn 0), cột 1 `[1,0,0,0]`, cột 2 `[1,1,0,0]`, cột 3 `[1,1,1,0]`; tổng hàng `[3, 2, 1, 0]`; nhà vô địch là đội `0`; Cách 3: `champ = 0` giữ nguyên với mọi `i` vì `grid[i][0] == 0`. Bài 3: ba dòng đã ghi trong comment. Bài 4: duyệt `a`, `beaten = any(grid[i][a] for i in range(n))`.
- **Mức 3 (sâu):** Bài 5(a): đội yếu nhất là đội không thắng ai: giữ ứng viên `w`; nếu `grid[w][i] == 1` (ứng viên thắng `i`) thì `w = i` (vì `w` không còn là yếu nhất). Bài 5(b): `[a for a in range(n) if all(grid[i][a] == 0 for i in range(n))]`; nếu độ dài bằng 1 thì trả phần tử, ngược lại `-1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
grid_demo = [[0, 0, 1], [1, 0, 1], [0, 0, 0]]

# Bài 1
assert grid_demo[1][2] == 1
assert [row[0] for row in grid_demo] == [0, 1, 0]
assert sum(grid_demo[1]) == 2
assert Solution().findChampion([[0, 0], [1, 0]]) == 1
assert Solution().findChampion([[0, 1, 1], [0, 0, 1], [0, 0, 0]]) == 0

# Bài 2
g4 = [[0, 1, 1, 1], [0, 0, 1, 1], [0, 0, 0, 1], [0, 0, 0, 0]]
assert [sum(r) for r in g4] == [3, 2, 1, 0]
assert [[g4[i][a] for i in range(4)] for a in range(4)][0] == [0, 0, 0, 0]
assert Solution().findChampion(g4) == 0

# Bài 3
def champion_todo_ans(grid):
    champ = 0
    for i in range(1, len(grid)):
        if grid[i][champ] == 1:
            champ = i
    return champ

for grid, expected in examples:
    assert champion_todo_ans(grid) == expected

# Bài 4
def champion_column_scan2(grid):
    n = len(grid)
    for a in range(n):
        if not any(grid[i][a] for i in range(n)):
            return a

for grid, expected in examples:
    assert champion_column_scan2(grid) == expected

# Bài 5(a)
def weakest_team(grid):
    w = 0
    for i in range(1, len(grid)):
        if grid[w][i] == 1:           # w thắng i -> w không phải yếu nhất; i là ứng viên mới
            w = i
    return w

g5 = make_grid_from_rank([3, 0, 4, 1, 2])
assert weakest_team(g5) == 2 and weakest_team(g4) == 3

# Bài 5(b)
def find_champion_or_minus_one(grid):
    n = len(grid)
    unbeaten = [a for a in range(n) if all(grid[i][a] == 0 for i in range(n))]
    return unbeaten[0] if len(unbeaten) == 1 else -1

assert find_champion_or_minus_one(grid_demo) == 1
assert find_champion_or_minus_one([[0, 0], [0, 0]]) == -1
assert find_champion_or_minus_one([[0, 1], [1, 0]]) == -1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ứng viên loại dần)

```python
from typing import List

class Solution:
    def findChampion(self, grid: List[List[int]]) -> int:
        champ = 0
        for i in range(1, len(grid)):
            if grid[i][champ] == 1:      # i mạnh hơn ứng viên
                champ = i
        return champ
```

**Checklist**

- [ ] Đã ghi rõ `grid[i][j] == 1` nghĩa là **`i` mạnh hơn `j`** (hàng thắng cột)?
- [ ] Nhà vô địch = **cột** toàn `0` (không ai thắng) hoặc hàng có `n - 1` số `1`, không nhầm với hàng toàn `0`?
- [ ] Khi loại ứng viên: đọc **`grid[i][champ]`** (i thắng champ), không phải `grid[champ][i]`?
- [ ] Trả về **chỉ số đội**, không phải số trận thắng?
- [ ] Hiểu cách `O(n)` dựa vào giả thiết phân định + có nhà vô địch?
- [ ] Đã thử `n = 2` (cả hai chiều), đội đầu / đội cuối là nhà vô địch?
- [ ] Skeleton đúng `class Solution` / `def findChampion(self, grid: List[List[int]]) -> int`?